# Emergency vs Non-Emergency Vehicle Detection — YOLO11 Fast & Stable

این نسخه از اول طوری نوشته شده که به **pandas، sklearn و matplotlib** وابسته نباشد.

دیتاست اصلی دست‌نخورده می‌ماند. از `data1` و `data2`، تمام تصاویر دارای کلاس `emergency` و به همان تعداد تصاویر `non_emergency` انتخاب می‌شوند. سپس داده‌ها به train/validation/test تقسیم شده و annotationهای polygon به bounding box استاندارد YOLO تبدیل می‌شوند.

مدل: **YOLO11n**، اندازه تصویر: **512**، حداکثر آموزش: **20 epoch** با Early Stopping.


## 1. بررسی محیط
این سل هیچ پکیج اضافی نصب نمی‌کند و فقط Python، PyTorch، CUDA و Ultralytics را بررسی می‌کند.

In [2]:
# CELL 01
import sys
import torch
import ultralytics

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available; training on CPU will be much slower.")


Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
Python executable: c:\Users\Amirmohammad\AppData\Local\Programs\Python\Python312\python.exe
PyTorch: 2.13.0+cu126
Ultralytics: 8.4.163
CUDA available: True
GPU: NVIDIA GeForce GTX 1660 Ti


## 2. کتابخانه‌های اصلی
عمداً `pandas`، `sklearn` و `matplotlib` اینجا استفاده نشده‌اند تا خطاهای قبلی باعث تعریف‌نشدن متغیرهای داده نشوند.

In [3]:
# CELL 02
from pathlib import Path
import random
import shutil
from collections import Counter

import cv2
import numpy as np
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Imports OK")


Imports OK


## 3. مسیر پروژه و دیتاست
مسیر Desktop معمولی و OneDrive Desktop هر دو بررسی می‌شوند.

In [4]:
# CELL 03
HOME = Path.home()
PROJECT_NAME = "Emergency and Non-Emergency Vehicle Detection Project"

CANDIDATES = [
    HOME / "Desktop" / PROJECT_NAME,
    HOME / "OneDrive" / "Desktop" / PROJECT_NAME,
]

PROJECT_DIR = next((p for p in CANDIDATES if p.is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project folder not found. Expected one of:\n" +
        "\n".join(str(p) for p in CANDIDATES)
    )

SOURCE_DIR = PROJECT_DIR / "emergency_dataset_with_label"
DATA_DIRS = [SOURCE_DIR / "data1", SOURCE_DIR / "data2"]

for d in DATA_DIRS:
    if not d.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {d}")

WORK_DIR = PROJECT_DIR / "yolo11_fast_working_dataset"
RUN_PROJECT_DIR = PROJECT_DIR / "yolo11_runs"

print("Project:", PROJECT_DIR)
print("Source dataset:", SOURCE_DIR)
print("Working dataset:", WORK_DIR)
print("Training runs:", RUN_PROJECT_DIR)


Project: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project
Source dataset: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\emergency_dataset_with_label
Working dataset: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset
Training runs: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_runs


## 4. Annotation conversion
در این دیتاست labelها polygon هستند. برای هر polygon، کمترین و بیشترین مختصات x و y گرفته می‌شود و به YOLO bounding box تبدیل می‌شود. اگر label از قبل ۵ مقدار استاندارد YOLO داشته باشد، همان نگه داشته می‌شود.

In [5]:
# CELL 04
CLASS_NAMES = ["emergency", "non_emergency"]
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def polygon_to_yolo_bbox(parts):
    if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
        return None

    try:
        cls_id = int(float(parts[0]))
        coords = [float(x) for x in parts[1:]]
    except ValueError:
        return None

    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    xc = max(0.0, min(1.0, (x_min + x_max) / 2.0))
    yc = max(0.0, min(1.0, (y_min + y_max) / 2.0))
    w = max(0.0, min(1.0, x_max - x_min))
    h = max(0.0, min(1.0, y_max - y_min))

    if cls_id not in (0, 1) or w <= 0 or h <= 0:
        return None

    return f"{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"

def convert_label(src, dst):
    out = []
    for raw in src.read_text(encoding="utf-8").splitlines():
        parts = raw.split()
        if not parts:
            continue

        # Standard YOLO bbox: class xc yc width height
        if len(parts) == 5:
            try:
                cls_id = int(float(parts[0]))
                vals = [max(0.0, min(1.0, float(x))) for x in parts[1:]]
                if cls_id in (0, 1) and vals[2] > 0 and vals[3] > 0:
                    out.append(f"{cls_id} {vals[0]:.6f} {vals[1]:.6f} {vals[2]:.6f} {vals[3]:.6f}")
            except ValueError:
                pass
        else:
            converted = polygon_to_yolo_bbox(parts)
            if converted is not None:
                out.append(converted)

    dst.write_text("\n".join(out), encoding="utf-8")
    return len(out)


## 5. ساخت فهرست تصاویر
هیچ DataFrame ساخته نمی‌شود. بنابراین خطای `train_df is not defined` در این نسخه اصلاً موضوعیت ندارد.

In [6]:
# CELL 05
items = []

for data_dir in DATA_DIRS:
    image_dir = data_dir / "images"
    label_dir = data_dir / "labels"

    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in VALID_EXTS:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue

        class_ids = set()
        object_count = 0
        for raw in label_path.read_text(encoding="utf-8").splitlines():
            p = raw.split()
            if not p:
                continue
            try:
                cls = int(float(p[0]))
            except ValueError:
                continue
            if cls in (0, 1):
                class_ids.add(cls)
                object_count += 1

        if object_count == 0:
            continue

        # Image-level tag for balancing/splitting.
        image_tag = 0 if 0 in class_ids else 1
        items.append({
            "image": image_path,
            "label": label_path,
            "tag": image_tag,
            "source": data_dir.name,
        })

counts = Counter(x["tag"] for x in items)
print("Usable images:", len(items))
print("Emergency images:", counts[0])
print("Non-emergency images:", counts[1])


Usable images: 1276
Emergency images: 248
Non-emergency images: 1028


## 6. انتخاب زیرمجموعه متعادل و سریع
تمام تصاویر emergency نگه داشته می‌شوند و به همان تعداد non-emergency انتخاب می‌شود. در نتیجه کلاس‌ها متعادل می‌شوند و حجم آموزش خیلی کمتر از کل دیتاست خواهد بود.

In [7]:
# CELL 06
emergency_items = [x for x in items if x["tag"] == 0]
non_emergency_items = [x for x in items if x["tag"] == 1]

rng = random.Random(SEED)
rng.shuffle(emergency_items)
rng.shuffle(non_emergency_items)

BALANCED_COUNT = min(len(emergency_items), len(non_emergency_items))
selected = emergency_items[:BALANCED_COUNT] + non_emergency_items[:BALANCED_COUNT]
rng.shuffle(selected)

print("Selected emergency:", sum(x["tag"] == 0 for x in selected))
print("Selected non-emergency:", sum(x["tag"] == 1 for x in selected))
print("Selected total:", len(selected))


Selected emergency: 248
Selected non-emergency: 248
Selected total: 496


## 7. تقسیم train / validation / test بدون sklearn
تقسیم به‌صورت stratified و با کد ساده انجام می‌شود: 70% train، 15% validation، 15% test.

In [8]:
# CELL 07
def stratified_split(records, train_ratio=0.70, val_ratio=0.15, seed=42):
    groups = {0: [], 1: []}
    for r in records:
        groups[r["tag"]].append(r)

    rng = random.Random(seed)
    train, val, test = [], [], []

    for cls in (0, 1):
        group = list(groups[cls])
        rng.shuffle(group)
        n = len(group)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train + n_val])
        test.extend(group[n_train + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test

train_items, val_items, test_items = stratified_split(selected, seed=SEED)

print("Train:", len(train_items), "| emergency:", sum(x["tag"] == 0 for x in train_items), "| non-emergency:", sum(x["tag"] == 1 for x in train_items))
print("Validation:", len(val_items), "| emergency:", sum(x["tag"] == 0 for x in val_items), "| non-emergency:", sum(x["tag"] == 1 for x in val_items))
print("Test:", len(test_items), "| emergency:", sum(x["tag"] == 0 for x in test_items), "| non-emergency:", sum(x["tag"] == 1 for x in test_items))


Train: 346 | emergency: 173 | non-emergency: 173
Validation: 74 | emergency: 37 | non-emergency: 37
Test: 76 | emergency: 38 | non-emergency: 38


## 8. ساخت دیتاست YOLO
دیتاست جدید در کنار دیتاست اصلی ساخته می‌شود؛ فایل‌های اصلی تغییر نمی‌کنند.

In [9]:
# CELL 08
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

for split in ("train", "val", "test"):
    (WORK_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (WORK_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def copy_split(records, split):
    copied = 0
    for index, item in enumerate(records):
        src_img = item["image"]
        src_lbl = item["label"]

        # Prefix avoids any filename collision between data1 and data2.
        stem = f"{item['source']}_{src_img.stem}_{index}"
        dst_img = WORK_DIR / "images" / split / f"{stem}{src_img.suffix.lower()}"
        dst_lbl = WORK_DIR / "labels" / split / f"{stem}.txt"

        shutil.copy2(src_img, dst_img)
        if convert_label(src_lbl, dst_lbl) > 0:
            copied += 1
        else:
            dst_img.unlink(missing_ok=True)
            dst_lbl.unlink(missing_ok=True)

    return copied

print("Copied train:", copy_split(train_items, "train"))
print("Copied validation:", copy_split(val_items, "val"))
print("Copied test:", copy_split(test_items, "test"))


Copied train: 346
Copied validation: 74
Copied test: 76


## 9. ساخت data.yaml

In [10]:
# CELL 09
DATA_YAML = WORK_DIR / 'data.yaml'
data_yaml_text = f'''path: {WORK_DIR.as_posix()}
train: images/train
val: images/val
test: images/test
names:
  0: emergency
  1: non_emergency
'''
DATA_YAML.write_text(data_yaml_text, encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))


path: C:/Users/Amirmohammad/Desktop/Emergency and Non-Emergency Vehicle Detection Project/yolo11_fast_working_dataset
train: images/train
val: images/val
test: images/test
names:
  0: emergency
  1: non_emergency



## 10. بررسی سریع annotationها
قبل از آموزش، چند label تبدیل‌شده خوانده می‌شود تا مطمئن شویم همه فایل‌های YOLO معتبرند.

In [11]:
# CELL 10
def validate_yolo_labels(split):
    label_dir = WORK_DIR / "labels" / split
    total = 0
    bad = 0
    for path in label_dir.glob("*.txt"):
        total += 1
        for line in path.read_text(encoding="utf-8").splitlines():
            p = line.split()
            if len(p) != 5:
                bad += 1
                continue
            try:
                cls = int(float(p[0]))
                vals = [float(x) for x in p[1:]]
                if cls not in (0, 1) or not all(0 <= v <= 1 for v in vals) or vals[2] <= 0 or vals[3] <= 0:
                    bad += 1
            except ValueError:
                bad += 1
    return total, bad

for split in ("train", "val", "test"):
    total, bad = validate_yolo_labels(split)
    print(split, "label files:", total, "invalid lines:", bad)
    if total == 0 or bad != 0:
        raise RuntimeError(f"Invalid YOLO labels detected in {split}.")

print("Annotation validation passed.")


train label files: 346 invalid lines: 0
val label files: 74 invalid lines: 0
test label files: 76 invalid lines: 0
Annotation validation passed.


## 11. تنظیمات آموزش
برای جلوگیری از آموزش چندساعته، از YOLO11n، تصویر 512، batch مثبت 4 و حداکثر 20 epoch استفاده می‌کنیم. `patience=5` اجازه می‌دهد آموزش زودتر متوقف شود.

In [12]:
# CELL 11
MODEL_NAME = "yolo11n.pt"
IMG_SIZE = 512
EPOCHS = 20
PATIENCE = 5
BATCH = 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
AMP = bool(torch.cuda.is_available())

print("Model:", MODEL_NAME)
print("Image size:", IMG_SIZE)
print("Epochs (maximum):", EPOCHS)
print("Patience:", PATIENCE)
print("Batch:", BATCH)
print("Device:", DEVICE)
print("AMP:", AMP)


Model: yolo11n.pt
Image size: 512
Epochs (maximum): 20
Patience: 5
Batch: 4
Device: 0
AMP: True


## 12. بارگذاری YOLO11n pretrained

In [13]:
# CELL 12
model = YOLO(MODEL_NAME)
print("Loaded:", MODEL_NAME)


Loaded: yolo11n.pt


## 13. آموزش
این سل فقط وقتی به مرحله آموزش می‌رسد که تمام سل‌های آماده‌سازی بالا بدون خطا اجرا شده باشند.

In [14]:
# CELL 13
RUN_NAME = "emergency_non_emergency_yolo11n_fast"

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    cache=False,
    amp=AMP,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=1.0,
    cos_lr=True,
    patience=PATIENCE,
    close_mosaic=3,
    mosaic=0.5,
    mixup=0.0,
    degrees=2.0,
    translate=0.08,
    scale=0.35,
    shear=1.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.25,
    pretrained=True,
    project=str(RUN_PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=False,
    verbose=True,
)

print("Training finished.")


New https://pypi.org/project/ultralytics/8.4.174 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.163  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset\data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.25, imgsz=512, iou=0.7, keras=False, kobj=1

## 14. بارگذاری بهترین وزن

In [15]:
# CELL 14
BEST_WEIGHTS = RUN_PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_WEIGHTS}")

best_model = YOLO(str(BEST_WEIGHTS))
print("Best weights:", BEST_WEIGHTS)


Best weights: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_runs\emergency_non_emergency_yolo11n_fast\weights\best.pt


## 15. Validation metrics
متریک‌های اصلی تشخیص را می‌گیریم: Precision، Recall، mAP50 و mAP50-95. F1 نیز از Precision و Recall محاسبه می‌شود.

In [16]:
# CELL 15
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    plots=False,
)

box = val_metrics.box
precision = float(box.mp)
recall = float(box.mr)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print("Validation Precision:", round(precision, 4))
print("Validation Recall:", round(recall, 4))
print("Validation F1:", round(f1, 4))
print("Validation mAP50:", round(float(box.map50), 4))
print("Validation mAP50-95:", round(float(box.map), 4))


Ultralytics 8.4.163  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 166.1104.4 MB/s, size: 21.1 KB)
val: Scanning C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset\labels\val.cache... 74 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 24.2it/s 0.8s0.1s
                   all         74         96      0.964      0.924      0.948      0.732
             emergency         37         45      0.954      0.926      0.941      0.735
         non_emergency         37         51      0.974      0.922      0.954      0.728
Speed: 0.3ms preprocess, 4.5ms inference, 0.0ms loss, 1.5ms postprocess per image
Validation Precision: 0.9639
Validation Reca

## 16. Test metrics — معیار نهایی پروژه

In [17]:
# CELL 16
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    plots=False,
)

box = test_metrics.box
precision = float(box.mp)
recall = float(box.mr)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
map50 = float(box.map50)
map5095 = float(box.map)

print("\n===== FINAL TEST METRICS =====")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))
print("mAP50:", round(map50, 4))
print("mAP50-95:", round(map5095, 4))

metrics_file = PROJECT_DIR / "final_test_metrics.txt"
metrics_file.write_text(
    "FINAL TEST METRICS\n"
    f"Precision: {precision:.4f}\n"
    f"Recall: {recall:.4f}\n"
    f"F1: {f1:.4f}\n"
    f"mAP50: {map50:.4f}\n"
    f"mAP50-95: {map5095:.4f}\n",
    encoding="utf-8"
)
print("Saved:", metrics_file)


Ultralytics 8.4.163  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 195.562.2 MB/s, size: 15.8 KB)
val: Scanning C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset\labels\test... 76 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 76/76 2.2Kit/s 0.0s
val: New cache created: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 26.3it/s 0.7s0.0s
                   all         76        100      0.945      0.885      0.958       0.73
             emergency         38         45      0.947      0.867      0.949       0.72
         non_emergency         39         55      0.943    

## 17. متریک‌های هر کلاس
اگر Ultralytics برای هر کلاس مقدار جداگانه گزارش کرده باشد، اینجا Precision/Recall/F1/mAP هر کلاس نمایش داده می‌شود.

In [18]:
# CELL 17
box = test_metrics.box
class_indices = list(box.ap_class_index)

for i, cls_idx in enumerate(class_indices):
    p = float(box.p[i])
    r = float(box.r[i])
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    ap50 = float(box.ap50[i])
    ap = float(box.ap[i])
    name = test_metrics.names[cls_idx]
    print(f"{name}: Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}, mAP50={ap50:.4f}, mAP50-95={ap:.4f}")


emergency: Precision=0.9468, Recall=0.8667, F1=0.9050, mAP50=0.9494, mAP50-95=0.7204
non_emergency: Precision=0.9431, Recall=0.9039, F1=0.9231, mAP50=0.9670, mAP50-95=0.7388


## 18. یک تست پیش‌بینی روی تصویر test
بدون matplotlib؛ نتیجه به صورت تصویر ذخیره می‌شود تا مشکل importهای قبلی دوباره وارد پروژه نشود.

In [19]:
# CELL 18
test_images = sorted((WORK_DIR / "images" / "test").glob("*"))
if not test_images:
    raise FileNotFoundError("No test images found.")

test_image = test_images[0]
pred = best_model.predict(
    source=str(test_image),
    conf=0.25,
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=False,
)

plotted = pred[0].plot()
out_path = PROJECT_DIR / "sample_test_prediction.jpg"
cv2.imwrite(str(out_path), plotted)

print("Input image:", test_image)
print("Prediction image saved:", out_path)
print("Detections:", len(pred[0].boxes))


Input image: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\yolo11_fast_working_dataset\images\test\data1_10e197e4-00-01-38_16H61420_full_27.jpg
Prediction image saved: C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\sample_test_prediction.jpg
Detections: 6


## 19. ذخیره مدل نهایی

In [20]:
# CELL 19
FINAL_MODEL = PROJECT_DIR / "emergency_non_emergency_yolo11n_best.pt"
shutil.copy2(BEST_WEIGHTS, FINAL_MODEL)

print("Final model saved:")
print(FINAL_MODEL)
print("\nProject completed successfully.")


Final model saved:
C:\Users\Amirmohammad\Desktop\Emergency and Non-Emergency Vehicle Detection Project\emergency_non_emergency_yolo11n_best.pt

Project completed successfully.


### ساختار خروجی
پس از اجرای کامل، این فایل‌ها/پوشه‌ها در پوشه پروژه ساخته می‌شوند: `yolo11_fast_working_dataset`، `yolo11_runs`، `emergency_non_emergency_yolo11n_best.pt`، `final_test_metrics.txt` و `sample_test_prediction.jpg`.